In [7]:
from automating_prisma.runtime import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index,
)

load_environment()

import json
import pandas as pd
import re

folder_path = data_directory()
result_path = results_directory()

file = result_path + 'CLAUDE_df_evaluated_validation.csv'

In [8]:
df_validation = pd.read_csv(file)
df_validation

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,yes,The article presents a new AQA dataset (MMW-AQ...,MMW-AQA
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...,no,The article is about Audio Question Answering ...,NaN
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,maybe,The article discusses action quality assessmen...,NaN
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,maybe,The article uses skeleton-based methods for ac...,NaN
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,maybe,The article proposes an AQA method evaluated o...,NaN
...,...,...,...,...,...,...,...,...
140,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,Automated vision-based score estimation models...,yes,Meets all inclusion criteria: uses Olympics Di...,NaN
141,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, Bogdan Emanuel; Seyerlehner, Klaus; M...",2014,"In this paper, we discuss and audio-visual app...",no,Not related to Human Action Quality Assessment...,NaN
142,10.1155/2017/9281321,A Multicriteria Intelligence Aid Methodology U...,"Frini, Anissa",2017,Intelligence is increasingly relevant today in...,no,Not related to Human Action Quality Assessment...,NaN
143,10.1038/s41558-018-0343-2,An assessment of climate action by high-carbon...,"Dietz, Simon; Fruitiere, Charles; Garcia-Manas...",2018,Corporations are an important source of GHG em...,no,Not related to Human Action Quality Assessment...,NaN


In [9]:
with open(folder_path + 'study_datasets.json', 'r') as f:
    study_papers_list = json.load(f)

study_papers_list

{'10.1145/2505323.2505330': 'Perturbed Workout; SU-41 Gesture',
 '10.1109/CVPRW.2013.82': 'Stroke Rehabilitation',
 '10.1109/EMBC.2014.6944292': 'self-created',
 '10.1007/978-3-319-10599-4_36': 'MIT-Dive; MIT-Skate',
 '10.1109/ICCT.2015.7399879': 'self-created',
 '10.1109/JBHI.2014.2375206': 'self-created',
 '10.1109/TPAMI.2016.2533388': 'Stroke Rehabilitation',
 '10.1109/CVPRW.2017.16': 'MIT-Dive; MIT-Skate; UNLV-Dive; UNLV-Vault',
 '10.1007/978-3-030-00767-6_12': 'UNLV-Dive; UNLV-Vault; MIT-Skate',
 '10.1109/ICIP.2018.8451364': 'UNLV-Dive',
 '10.1109/CVPR.2019.00039': 'MTL-AQA',
 '10.1007/978-3-030-20876-9_10': 'MIT-Dive; UNLV-Dive; UNLV-Vault',
 '10.1109/ICCV.2019.00643': 'AQA-7; JIGSAWS',
 '10.1109/WACV.2019.00161': 'AQA-7',
 '10.1016/j.compbiomed.2020.104021': 'self-created',
 '10.1109/TransAI49837.2020.00030': 'UI-PRMD',
 '10.1145/3341105.3374092': 'Origami Video',
 '10.1109/CVPR42600.2020.00986': 'AQA-7; MTL-AQA; JIGSAWS',
 '10.1145/3394171.3413560': 'Rhythmic Gymnastics; MIT-Sk

In [10]:
verified_dois_study = set(study_papers_list.keys())
verified_dois_validation = set(df_validation["DOI"].tolist())

matches = verified_dois_study & verified_dois_validation
num_matches = len(matches)

only_study = verified_dois_study - verified_dois_validation
num_study = len(only_study)

only_validation = verified_dois_validation - verified_dois_study
num_validation = len(only_validation)

print("Perfect Matches:", num_matches)
print("Only in Study:", num_study)
print("Only in Validation:", num_validation)

Perfect Matches: 70
Only in Study: 24
Only in Validation: 75


In [11]:
df_validation = df_validation[df_validation["Include"] != "no"].reset_index(drop=True)
df_validation

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,yes,The article presents a new AQA dataset (MMW-AQ...,MMW-AQA
1,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,maybe,The article discusses action quality assessmen...,NaN
2,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,maybe,The article uses skeleton-based methods for ac...,NaN
3,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,maybe,The article proposes an AQA method evaluated o...,NaN
4,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,"In recent years, there has been growing intere...",yes,The article presents a Multi-stage Contrastive...,NaN
...,...,...,...,...,...,...,...,...
90,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,This paper presents a dataset for vision-based...,yes,Meets all inclusion criteria: presents a multi...,FMS
91,10.3390/en15062071,Manual Operation Evaluation Based on Vectorize...,"He, Fangqiuzi; Liu, Yong; Zhan, Weiwen; Xu, Qi...",2022,The standard of manual operation in smart grid...,yes,Meets all inclusion criteria: uses datasets fr...,NaN
92,10.1155/2022/8950732,A Data Mining-Based Model for Evaluating Tenni...,"Chen, Hang",2022,This paper uses data mining technology to math...,yes,Meets all inclusion criteria: establishes a 3D...,NaN
93,10.1155/2022/5430463,RETRACTED: Research on Tennis Motion Evaluatio...,"Chang, Huan",2022,The current tennis training lacks the guidance...,yes,Meets all inclusion criteria: uses MSR Action ...,MSR Action 3D


In [12]:
df_validation.to_csv(result_path + "CLAUDE_df_screening_passed.csv", index=False)

### Total of 95 papers identified against 101 of the original study

In [6]:
df_validation_reviewed = (
    df_validation[
        (df_validation["Data Source"].notna()) &
        (df_validation["Data Source"].astype(str).str.strip() != "")
    ]
    .reset_index(drop=True)
)

df_validation_reviewed = df_validation_reviewed[["DOI", "Title", "Authors", "Year", "Data Source"]]

dataset_map = {
    "AQA": ["AQA-7"],
    "MIT-Skating": ["MIT-Skate"],
    "MIT Figure Skating": ["MIT-Skate"],
    "MIT-Diving": ["MIT-Dive"],
    "Olympics Diving": ["MIT-Dive"],
    "MIT Olympic Scoring": ["MIT-Skate", "MIT-Dive"],
    "Olympic Sports": ["MIT-Skate", "MIT-Dive"],
    "UNLV-Skating": ["UNLV-Skate"],
    "UNLV-Diving": ["UNLV-Dive"],
    "Gymnastic vaults": ["UNLV-Vault"],
    "UNLV Olympic Scoring": ["UNLV-Skate", "UNLV-Dive", "UNLV-Vault"],
    "FineDiving-HM": ["FineDiving"],
    "Fis-V": ["FIS-V"],
    "Fitness Action": ["Fitness-AQA"],
    "IntelliRehabDS": ["IRDS"],
    "JIGSAWSS": ["JIGSAWS"],
    "MSR Action 3D": ["MSR-Action3D"],
    "MSRC12": ["MSRC-12"],
    "Multitask-AQA": ["MTL-AQA"],
    "Play Therapy 13 (PT13)": ["PT13"],

    "PSA": ["PISA"],
    "FR-FS": ["MIT-Skate"],
    "Pull-ups": ["self-created"],
    "FMS": ["self-created"]
}


def map_partial_sources(cell):
    if pd.isna(cell):
        return cell

    parts = [p.strip() for p in str(cell).split(";") if p.strip()]
    out = []

    for p in parts:
        matched = False
        for k, v in dataset_map.items():
            pattern = fr"(^|;){re.escape(k)}($|;)"
            if re.fullmatch(k, p):
                out.extend(v)
                matched = True
                break
            elif re.search(pattern, ";" + p + ";"):
                out.extend(v)
                matched = True
                break
        if not matched:
            out.append(p)

    seen = set()
    result = []
    for x in out:
        key = x.strip()
        if key not in seen:
            seen.add(key)
            result.append(key)

    result = [x for x in result if len(x) <= 20]

    return ";".join(result)

df_validation_reviewed["Data Source"] = df_validation_reviewed["Data Source"].apply(map_partial_sources)

df_validation_reviewed["Data Source"] = df_validation_reviewed["Data Source"].replace("", pd.NA)
df_validation_reviewed["Data Source"] = df_validation_reviewed["Data Source"].fillna("self-created")

df_validation_reviewed.to_csv(result_path + "CLAUDE_review.csv", index=False)
df_validation_reviewed

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,MMW-AQA
1,10.1109/CVPR52729.2023.00238,LOGO: A Long-Form Video Dataset for Group Acti...,S. Zhang; W. Dai; S. Wang; X. Shen; J. Lu; J. ...,2023,LOGO
2,10.1109/TIM.2024.3398072,Learning Sparse Temporal Video Mapping for Act...,S. Zahan; G. Mubashar Hassan; A. Mian,2024,AGF-Olympics
3,10.1109/CVPR52688.2022.00296,FineDiving: A Fine-grained Dataset for Procedu...,J. Xu; Y. Rao; X. Yu; G. Chen; J. Zhou; J. Lu,2022,FineDiving
4,10.1109/WACV57701.2024.00012,PECoP: Parameter Efficient Continual Pretraini...,A. Dadashzadeh; S. Duan; A. Whone; M. Mirmehdi,2024,JIGSAWS;MTL-AQA;FineDiving;PD4T
...,...,...,...,...,...
61,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,self-created
62,10.1007/978-3-319-10599-4_36,Assessing the quality of actions,"Pirsiavash, H.; Vondrick, C.; Torralba, A.",2014,MIT-Skate;MIT-Dive
63,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,AQA-7;MTL-AQA;JIGSAWS
64,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,self-created


In [40]:
verified_dois_study = set(study_papers_list.keys())
verified_dois_validation = set(df_validation["DOI"].tolist())

matches = verified_dois_study & verified_dois_validation
num_matches = len(matches)

only_study = verified_dois_study - verified_dois_validation
num_study = len(only_study)

only_validation = verified_dois_validation - verified_dois_study
num_validation = len(only_validation)

print("Perfect Matches:", num_matches)
print("Only in Study:", num_study)
print("Only in Validation:", num_validation)

Perfect Matches: 69
Only in Study: 25
Only in Validation: 26


In [41]:
df_final = df_validation[df_validation["Data Source"].isna()].reset_index(drop=True)
df_final

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,maybe,The article discusses action quality assessmen...,NaN
1,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,maybe,The article uses skeleton-based methods for ac...,NaN
2,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,maybe,The article proposes an AQA method evaluated o...,NaN
3,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,"In recent years, there has been growing intere...",yes,The article presents a Multi-stage Contrastive...,NaN
4,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,Given the current increasing prevalence of aut...,maybe,The article uses publicly available autism dat...,NaN
5,10.1109/ICIP.2018.8451364,S3D: Stacking Segmental P3D for Action Quality...,X. Xiang; Y. Tian; A. Reiter; G. D. Hager; T. ...,2018,Action quality assessment is crucial in areas ...,maybe,The article proposes a method for action quali...,NaN
6,10.1109/TIP.2024.3362135,Multimodal Action Quality Assessment,L. -A. Zeng; W. -S. Zheng,2024,Action quality assessment (AQA) is to assess h...,yes,The article proposes a multimodal action quali...,NaN
7,10.1109/TCSVT.2022.3143549,Semi-Supervised Action Quality Assessment With...,S. -J. Zhang; J. -H. Pan; J. Gao; W. -S. Zheng,2022,Action Quality Assessment aims to evaluate how...,maybe,The article proposes a semi-supervised framewo...,NaN
8,10.1109/ICCT.2015.7399879,Virtual Personal Trainer via the Kinect Sensor,Xin Jin; Yuan Yao; Qiliang Jiang; Xingying Hua...,2015,Our daily life is more convenient than ever be...,maybe,The paper proposes a Virtual Personal Trainer ...,NaN
9,10.1109/CVPR52688.2022.00323,Likert Scoring with Grade Decoupling for Long-...,A. Xu; L. -A. Zeng; W. -S. Zheng,2022,Long-term action quality assessment is a task ...,yes,The paper proposes a Grade-decoupling Likert T...,NaN


In [42]:
df_final.to_csv(result_path + "CLAUDE_df_evaluated_validation_abstract.csv", index=False)

In [43]:
unique_sources = df_validation['Data Source'].dropna().unique().tolist()
unique_sources = [source.strip() for item in unique_sources for source in item.split(';')]
unique_sources = set(unique_sources)

filtered_sources = set()

for source in unique_sources:
    mapped = dataset_map.get(source, [source])
    filtered_sources.update([s for s in mapped if len(s) <= 20])

filtered_sources.discard("self-created")

print(f"The total number of normalized sources in Validation is: {len(filtered_sources)}\n")
sorted(filtered_sources, key=len)

The total number of normalized sources in Validation is: 44



['JDM',
 'KTH',
 'EHE',
 'LOGO',
 'PaSk',
 'PT13',
 'PISA',
 'TAQR',
 'IRDS',
 'RFSJ',
 'PD4T',
 'NETS',
 'RDSD',
 'FIS-V',
 'MMASD',
 'AQA-7',
 'KIMORE',
 'FineFS',
 'TASD-2',
 'JIGSAWS',
 'FineGym',
 'MMW-AQA',
 'MSRC-12',
 'MTL-AQA',
 'UI-PRMD',
 'Thumos14',
 'MIT-Dive',
 'TaiChi-24',
 'MIT-Skate',
 'FineRehab',
 'OlympicFS',
 'UNLV-Dive',
 'UNLV-Skate',
 'UNLV-Vault',
 'FineDiving',
 'Fitness-28',
 'Fitness-AQA',
 'Walking Gait',
 'MSR-Action3D',
 'UMONS-TAICHI',
 'AGF-Olympics',
 'Origami Video',
 'ActivityNet1.3',
 'Rhythmic Gymnastics']

In [44]:
verified_datasets_study = set()

for value in study_papers_list.values():
    if value:
        parts = [v.strip() for v in value.split(';') if v.strip()]
        verified_datasets_study.update(parts)

verified_datasets_study.discard("self-created")

print(f"The total number of unique datasets in study_papers_list: {len(verified_datasets_study)}\n")
sorted(verified_datasets_study, key=len)

The total number of unique datasets in study_papers_list: 47



['JDM',
 'EHE',
 'LOGO',
 'PaSk',
 'PT13',
 'PISA',
 'BEST',
 'RFSJ',
 'IRDS',
 'PD4T',
 'NETS',
 'DREAM',
 'FIS-V',
 'MMASD',
 'Squat',
 'AQA-7',
 'KIMORE',
 'FineFS',
 'TASD-2',
 'FS1OOO',
 'JIGSAWS',
 'Fri2023',
 'FineGym',
 'MMW-AQA',
 'MTL-AQA',
 'UI-PRMD',
 'MIT-Dive',
 'TaiChi-24',
 'MIT-Skate',
 'CPR-Coach',
 'TGC20ReId',
 'OlympicFS',
 'UNLV-Dive',
 'UNLV-Skate',
 'UNLV-Vault',
 'FineDiving',
 'Fitness-28',
 'EPIC-Skills',
 'Walking Gait',
 'SkatingVerse',
 'UMONS-TAICHI',
 'SU-41 Gesture',
 'Origami Video',
 'AI-Hub Fitness',
 'Perturbed Workout',
 'Rhythmic Gymnastics',
 'Stroke Rehabilitation']

In [45]:
# Elements in common
common_elements = filtered_sources & verified_datasets_study

# Elements unique to each list
only_in_filtered = filtered_sources - verified_datasets_study
only_in_original = verified_datasets_study - filtered_sources


print("Elements in common:", common_elements)
print("Number of elements in common:", len(common_elements))

print("\nElements in Validation but not in Study:", only_in_filtered)
print("Number:", len(only_in_filtered))

print("\nElements in Study but not in Validation:", only_in_original)
print("Number:", len(only_in_original))

Elements in common: {'Rhythmic Gymnastics', 'JIGSAWS', 'LOGO', 'TaiChi-24', 'MIT-Skate', 'PaSk', 'MIT-Dive', 'Walking Gait', 'PT13', 'FineGym', 'PISA', 'FIS-V', 'MMASD', 'OlympicFS', 'MMW-AQA', 'KIMORE', 'FineFS', 'MTL-AQA', 'UI-PRMD', 'JDM', 'IRDS', 'RFSJ', 'PD4T', 'UNLV-Skate', 'UNLV-Dive', 'NETS', 'UMONS-TAICHI', 'EHE', 'TASD-2', 'UNLV-Vault', 'FineDiving', 'Origami Video', 'Fitness-28', 'AQA-7'}
Number of elements in common: 34

Elements in Validation but not in Study: {'MSRC-12', 'TAQR', 'Fitness-AQA', 'KTH', 'RDSD', 'Thumos14', 'AGF-Olympics', 'FineRehab', 'ActivityNet1.3', 'MSR-Action3D'}
Number: 10

Elements in Study but not in Validation: {'Stroke Rehabilitation', 'Fri2023', 'Squat', 'DREAM', 'SU-41 Gesture', 'FS1OOO', 'EPIC-Skills', 'Perturbed Workout', 'CPR-Coach', 'SkatingVerse', 'TGC20ReId', 'BEST', 'AI-Hub Fitness'}
Number: 13
